In [0]:
# Databricks Notebook: Loan_Summary
# Cell 1: Aggregate Loan Exposure by Type and Status

from pyspark.sql.functions import col, count, current_timestamp, sum

df_loan = spark.table("bankingpoc.silver.loan")

df_loan_metrics = df_loan.groupBy("loan_type", "loan_status").agg(
    count("loan_id").alias("total_loans_issued"),
    sum("loan_amount").cast("decimal(18,2)").alias("total_disbursed_amount"),
    sum("paid_amount").cast("decimal(18,2)").alias("total_recovered_amount"),
    sum(col("loan_amount") - col("paid_amount")).cast("decimal(18,2)").alias("total_outstanding_amount")
).withColumn("gold_processed_timestamp", current_timestamp())

(df_loan_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bankingpoc.gold.loan_summary"))

print("Gold table bankingpoc.gold.loan_summary successfully generated.")

Gold table bankingpoc.gold.loan_summary successfully generated.
